<a href="https://colab.research.google.com/github/kathjeonbt-sys/Inteligencia-Artificial-II/blob/main/Sesion08-Modelo_prediccion_casas_Housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo predictivo para estimar el precio de viviendas

En este proyecto voy a comparar dos modelos de Machine Learning: **Regresión Lineal** y **Árbol de Decisión**. El objetivo es estimar el precio de una vivienda usando las características que aparecen en `Housing.csv`.

In [ ]:
# Importo pandas porque lo voy a usar para leer y organizar los datos como una tabla.
import pandas as pd

# Importo numpy porque lo necesito para calcular la raíz cuadrada del error.
import numpy as np

# Importo matplotlib porque quiero mostrar los resultados mediante gráficos.
import matplotlib.pyplot as plt

# Importo la función que me permite separar los datos para entrenar y probar los modelos.
from sklearn.model_selection import train_test_split

# Importo estas herramientas para preparar las columnas de texto y las columnas numéricas.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Importo el modelo de regresión lineal que voy a utilizar como primera opción.
from sklearn.linear_model import LinearRegression

# Importo el árbol de decisión para compararlo con la regresión lineal.
from sklearn.tree import DecisionTreeRegressor

# Importo las métricas que me permiten medir qué tan buenos son los modelos.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importo Pipeline para unir la preparación de los datos con cada modelo de una forma ordenada.
from sklearn.pipeline import Pipeline

In [ ]:
# Leo el archivo Housing.csv que subí a Google Colab.
datos = pd.read_csv('Housing.csv')

# Muestro las primeras cinco filas para comprobar que el archivo se cargó correctamente.
datos.head()

In [ ]:
# Muestro la cantidad de filas y columnas para conocer el tamaño del dataset.
print('Tamaño del dataset:', datos.shape)

# Reviso los tipos de datos porque algunas columnas contienen números y otras palabras.
print(datos.dtypes)

# Cuento los valores vacíos para comprobar si necesito limpiar datos faltantes.
print('\nValores faltantes por columna:')
print(datos.isnull().sum())

In [ ]:
# Elimino filas incompletas para trabajar solamente con registros que tengan todos los datos necesarios.
datos = datos.dropna()

# Muestro nuevamente el tamaño para comprobar cuántos registros quedaron después de la limpieza.
print('Tamaño después de limpiar:', datos.shape)

In [ ]:
# Separo las características de las viviendas de la columna price, que es el valor que quiero predecir.
X = datos.drop(columns=['price'])

# Guardo price como la variable objetivo porque representa el precio real de cada vivienda.
y = datos['price']

# Identifico las columnas que tienen texto porque debo convertirlas a números antes de entrenar los modelos.
columnas_texto = X.select_dtypes(include='object').columns

# Identifico las columnas numéricas que ya están listas para ser utilizadas por los modelos.
columnas_numericas = X.select_dtypes(exclude='object').columns

# Muestro qué columnas voy a tratar como texto y cuáles como números.
print('Columnas de texto:', list(columnas_texto))
print('Columnas numéricas:', list(columnas_numericas))

In [ ]:
# Convierto las categorías de texto en valores que los modelos puedan interpretar sin inventar un orden entre categorías.
preparador = ColumnTransformer([
    ('texto', OneHotEncoder(handle_unknown='ignore'), columnas_texto)
], remainder='passthrough')

# Separo el 80% de los datos para aprender y dejo el 20% para comprobar las predicciones.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Muestro cuántos registros quedaron para entrenar y para probar.
print('Datos de entrenamiento:', X_train.shape)
print('Datos de prueba:', X_test.shape)

In [ ]:
# Creo la regresión lineal porque quiero comprobar qué tan bien funciona un modelo que busca relaciones directas entre las variables.
modelo_lineal = Pipeline([
    ('preparacion', preparador),
    ('modelo', LinearRegression())
])

# Entreno la regresión lineal usando solamente los datos de entrenamiento.
modelo_lineal.fit(X_train, y_train)

# Pido al modelo que estime los precios de las viviendas que reservé para probarlo.
pred_lineal = modelo_lineal.predict(X_test)

In [ ]:
# Calculo el error absoluto promedio para saber cuánto se alejan las predicciones del precio real en promedio.
mae_lineal = mean_absolute_error(y_test, pred_lineal)

# Calculo el RMSE porque penaliza más los errores grandes y me ayuda a detectar predicciones muy alejadas.
rmse_lineal = np.sqrt(mean_squared_error(y_test, pred_lineal))

# Calculo R2 para conocer qué proporción de la variación de los precios logra explicar el modelo.
r2_lineal = r2_score(y_test, pred_lineal)

# Muestro las tres métricas para poder compararlas después con el árbol de decisión.
print('REGRESIÓN LINEAL')
print('MAE:', mae_lineal)
print('RMSE:', rmse_lineal)
print('R2:', r2_lineal)

In [ ]:
# Creo un árbol de decisión para comprobar si un modelo basado en divisiones de los datos puede predecir mejor.
modelo_arbol = Pipeline([
    ('preparacion', preparador),
    ('modelo', DecisionTreeRegressor(max_depth=5, random_state=42))
])

# Entreno el árbol usando los mismos datos de entrenamiento para que la comparación sea justa.
modelo_arbol.fit(X_train, y_train)

# Obtengo los precios que el árbol calcula para los datos de prueba.
pred_arbol = modelo_arbol.predict(X_test)

In [ ]:
# Calculo el error absoluto promedio del árbol para compararlo con el error de la regresión.
mae_arbol = mean_absolute_error(y_test, pred_arbol)

# Calculo el RMSE del árbol para revisar especialmente los errores grandes.
rmse_arbol = np.sqrt(mean_squared_error(y_test, pred_arbol))

# Calculo R2 para saber cuánto logra explicar el árbol de los cambios en los precios.
r2_arbol = r2_score(y_test, pred_arbol)

# Muestro los resultados del segundo modelo.
print('ÁRBOL DE DECISIÓN')
print('MAE:', mae_arbol)
print('RMSE:', rmse_arbol)
print('R2:', r2_arbol)

In [ ]:
# Creo una tabla con los resultados para que la comparación entre modelos sea más clara.
resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Árbol de Decisión'],
    'MAE': [mae_lineal, mae_arbol],
    'RMSE': [rmse_lineal, rmse_arbol],
    'R2': [r2_lineal, r2_arbol]
})

# Muestro la tabla final de comparación.
resultados

In [ ]:
# Creo un gráfico para comparar visualmente el R2 de los dos modelos.
plt.figure(figsize=(8, 5))

# Coloco cada modelo en el eje horizontal y su R2 en el eje vertical.
plt.bar(resultados['Modelo'], resultados['R2'])

# Agrego un título para explicar qué estoy comparando.
plt.title('Comparación del rendimiento de los modelos')

# Indico que el eje vertical muestra el valor de R2.
plt.ylabel('R2')

# Muestro el gráfico para poder analizarlo.
plt.show()

In [ ]:
# Comparo los errores RMSE para identificar cuál modelo realiza predicciones más cercanas.
if rmse_lineal < rmse_arbol:
    # Si la regresión tiene menor RMSE, la considero la mejor de las dos en esta prueba.
    mejor_modelo = 'Regresión Lineal'
else:
    # Si el árbol tiene menor RMSE, considero que obtuvo el mejor resultado.
    mejor_modelo = 'Árbol de Decisión'

# Muestro cuál modelo tuvo el menor error en los datos de prueba.
print('El modelo con menor RMSE fue:', mejor_modelo)

In [ ]:
# Creo una vivienda de ejemplo con características que puedo cambiar para probar el modelo.
casa_nueva = pd.DataFrame({
    'area': [5000],
    'bedrooms': [3],
    'bathrooms': [2],
    'stories': [2],
    'mainroad': ['yes'],
    'guestroom': ['no'],
    'basement': ['yes'],
    'hotwaterheating': ['no'],
    'airconditioning': ['yes'],
    'parking': [2],
    'prefarea': ['yes'],
    'furnishingstatus': ['semi-furnished']
})

# Calculo el precio estimado por la regresión lineal para esta vivienda nueva.
precio_lineal = modelo_lineal.predict(casa_nueva)[0]

# Calculo el precio estimado por el árbol para la misma vivienda.
precio_arbol = modelo_arbol.predict(casa_nueva)[0]

# Muestro las dos estimaciones para observar cómo cambia el resultado según el modelo.
print(f'Precio estimado con Regresión Lineal: ${precio_lineal:,.0f}')
print(f'Precio estimado con Árbol de Decisión: ${precio_arbol:,.0f}')

## Análisis de los resultados

Con los datos de prueba, la **Regresión Lineal obtuvo un MAE aproximado de 970.043**, lo que significa que sus predicciones se alejaron del precio real en alrededor de esa cantidad en promedio. Su **RMSE fue de aproximadamente 1.324.507** y su **R² fue de 0,653**.

El **Árbol de Decisión obtuvo un MAE aproximado de 1.199.910**, un **RMSE de aproximadamente 1.616.167** y un **R² de 0,483**. Por estas métricas, en esta prueba la Regresión Lineal tuvo un mejor rendimiento: presentó menores errores y un R² más alto.

Esto no significa que la Regresión Lineal siempre sea mejor que un Árbol de Decisión. El resultado depende de los datos, las variables utilizadas y la configuración del modelo. En este caso, la regresión logró representar mejor la relación existente entre las características de las viviendas y su precio.

## Conclusión

Los dos modelos permitieron estimar el precio de las viviendas a partir de características como el área, número de habitaciones, baños, pisos, estacionamiento y otras condiciones de la casa. Después de comparar sus resultados, la **Regresión Lineal fue el modelo con mejor rendimiento en este experimento**, porque tuvo menor error y un R² superior al Árbol de Decisión. Por lo tanto, para este dataset sería la opción que elegiría para realizar las predicciones, aunque sería posible probar otros modelos y configuraciones para intentar mejorar el resultado.